# 01 — Data Ingestion

This notebook walks through SEC EDGAR data retrieval and market price ingestion
for the NVDA Quantamental Engine. It demonstrates:

1. **Configuration** — how `EngineConfig` parameterises the pipeline
2. **SEC submissions** — fetching filing metadata with availability-date filtering
3. **Company facts** — retrieving structured XBRL data
4. **Filing documents** — downloading full 10-K / 10-Q HTML
5. **Market prices** — adjusted close with LOCF and staleness controls
6. **Peer financials** — EV inputs with staleness checks
7. **Provenance logging** — every download is recorded

All functions live in `src/edgar_fetch.py` and `src/config.py`.
The caching layer means re-running this notebook is fast — it skips
already-downloaded files unless `--force-refresh` is set.

> **Note:** This notebook requires network access for the initial run.
> Subsequent runs use cached data in `data/raw/`.

> **Note:** The authoritative reproducibility path is `python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01 --output-format both`. This notebook is an explanatory wrapper that inspects the same pipeline outputs.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.config import EngineConfig, get_default_config
from src.edgar_fetch import EdgarFetcher

## 1. Configuration

`EngineConfig` holds every parameter: ticker, CIK, fiscal-year range,
frozen `report_date` / `price_date`, peer lists, and path defaults.
Switching to a different company requires only config changes.

In [ ]:
config = get_default_config()

print(f"Ticker:       {config.ticker}")
print(f"CIK:          {config.cik}")
print(f"Report date:  {config.report_date}")
print(f"Price date:   {config.price_date}")
print(f"Fiscal years: {config.start_fiscal_year}–{config.end_fiscal_year}")
print(f"Peers (semi): {config.core_semiconductor_peers}")
print(f"Peers (infra): {config.infrastructure_peers}")
print(f"Peers (capex): {config.ai_capex_context}")

## 2. SEC Submissions

`fetch_submissions()` hits the SEC submissions endpoint, filters to
10-K and 10-Q filings within the fiscal-year range, and enforces
`source_available_date <= report_date`. Results are cached as JSON.

In [ ]:
fetcher = EdgarFetcher(config)

submissions = fetcher.fetch_submissions(config.cik)
print(f"Retrieved {len(submissions)} filings")
submissions.head(10)

Every row carries `source_available_date` (= `filing_date` for SEC filings).
This is the universal availability gate — no data enters the report or ML
model unless its availability date passes the cutoff.

## 3. Company Facts (XBRL)

`fetch_companyfacts()` retrieves the full structured XBRL dataset.
Each fact's `filed` date is annotated as `source_available_date`.

In [ ]:
facts = fetcher.fetch_companyfacts(config.cik)

# Show available taxonomies and a sample concept
taxonomies = list(facts.get("facts", {}).keys())
print(f"Taxonomies: {taxonomies}")

# Count total concepts
total_concepts = sum(
    len(tax_data) for tax_data in facts.get("facts", {}).values()
)
print(f"Total XBRL concepts available: {total_concepts}")

## 4. Filing Documents

`fetch_filing_document()` downloads the full HTML for a specific
accession number. This HTML is later parsed for narrative sections
(Risk Factors, MD&A, etc.).

In [ ]:
# Fetch the most recent 10-K as an example
if not submissions.empty:
    ten_k = submissions[submissions["form_type"] == "10-K"]
    if not ten_k.empty:
        latest_10k = ten_k.iloc[0]
        print(f"Fetching 10-K: {latest_10k['accession_number']}")
        print(f"  Filing date: {latest_10k['filing_date']}")
        print(f"  Report period: {latest_10k['report_period']}")
        html = fetcher.fetch_filing_document(
            latest_10k["accession_number"], config.cik
        )
        print(f"  HTML length: {len(html):,} characters")
    else:
        print("No 10-K filings found in submissions.")
else:
    print("No submissions available — run with network access first.")

## 5. Market Prices

`fetch_market_prices()` retrieves daily adjusted close via yfinance.
LOCF (last observation carried forward) fills weekend/holiday gaps
up to 3 calendar days. Longer gaps are logged and excluded.

In [ ]:
all_tickers = [config.ticker] + config.core_semiconductor_peers
prices = fetcher.fetch_market_prices(
    tickers=all_tickers,
    start="2016-01-01",
    end=config.price_date,
)
print(f"Price rows: {len(prices)}")
print(f"Tickers: {prices['ticker'].unique().tolist()}")
prices.groupby("ticker")["adj_close"].last()

## 6. Peer Financials

`fetch_peer_financials()` collects EV inputs (market cap, debt, cash),
revenue, EBITDA, net income, and FCF for each peer. A staleness check
flags peers whose data is older than the configured threshold (90 days).
EV-based multiples are skipped for stale peers.

In [ ]:
all_peers = (
    config.core_semiconductor_peers
    + config.infrastructure_peers
    + config.ai_capex_context
)
peer_fin = fetcher.fetch_peer_financials(all_peers)
print(f"Peer financial rows: {len(peer_fin)}")
print(f"Stale EV flags: {peer_fin['stale_ev'].sum()} of {len(peer_fin)}")
peer_fin[["ticker", "market_cap", "revenue", "stale_ev"]]

## 7. Provenance Log

Every download is recorded in `data/raw/provenance_log.jsonl`.
Each entry includes timestamp, step, source URL, and metadata.

In [ ]:
import json
from pathlib import Path

log_path = Path(config.provenance_log)
if log_path.exists():
    with open(log_path) as f:
        entries = [json.loads(line) for line in f if line.strip()]
    print(f"Provenance entries: {len(entries)}")
    # Show the last 3 entries
    for entry in entries[-3:]:
        print(f"  [{entry.get('step')}] {entry.get('source_url', '')[:80]}")
else:
    print("No provenance log yet — run the pipeline first.")

---

**Next:** [02_parsing_and_segments.ipynb](02_parsing_and_segments.ipynb) —
XBRL parsing, validation, segment normalization, and text extraction.